# 📘 Khóa học: HR Analyst với SQL + PostgreSQL (7 buổi)

## 🎯 Tổng quan chương trình

- **Buổi 1:** 🚀 Giới thiệu dự án
- **Buổi 2:** 🗄️ Schema & Setup database
- **Buổi 3:** 👥 Employee Overview Analysis
- **Buổi 4:** 📊 Turnover Analysis
- **Buổi 5:** 💰 Salary Analysis
- **Buổi 6:** 📈 Engagement & Advanced Analysis
- **Buổi 7:** ✅ Quiz HR Analytics SQL

---

# 📈 Buổi 6: Engagement & Advanced Analysis

## 🎯 Mục tiêu buổi này
✅ Phân tích Engagement Score và Wellbeing Score

✅ Xác định Employee Risk Group (churn prediction)

✅ Sử dụng Advanced Window Functions (RANK, ROW_NUMBER, LAG) và CTEs

✅ Tính NPS (Net Promoter Score) - khách hàng nội bộ

✅ Thực hành 5 bài tập cơ bản + 4 bài nâng cao với đáp án chi tiết

✅ Ôn tập 5 quiz từ cơ bản → nâng cao

---

## 🌏 Đề tài Engagement & Advanced Analysis là gì?

**Engagement & Wellbeing Analysis** là phân tích mức độ gắn bó, sức khỏe tinh thần của nhân viên. Đây là chủ đề "hot" năm 2026, giúp doanh nghiệp:

- 🧠 **Phát hiện burnout sớm**: Ngả máu nhân tài trước khi họ đi
- 😊 **Cải thiện workplace culture**: Tăng morale, giảm stress
- 🔮 **Dự đoán churn**: Engagement thấp = nguy cơ nghỉ việc cao
- 📊 **So sánh phòng ban**: Dept nào cần support ngay

### 📊 Ví dụ điển hình ở Việt Nam (2026)
- **Ngành Tech**: Engagement 6.2/10 (burnout từ AI competition)
- **Remote-first**: Wellbeing 5.8/10 (isolation, meeting fatigue)
- **Hybrid**: Engagement cao hơn 0.5 điểm vs full onsite
- **NPS nội bộ**: -15 (nhiều nhân viên sẵn sàng bỏ việc)

**Tác động kinh tế**: Nhân viên engagement +1 điểm = tăng productivity 3%, giảm turnover 5%.

### ❓ Các truy vấn thường được hỏi
- Engagement score trung bình theo phòng ban?
- Nhân viên nào ở nhóm "high risk" (engagement + wellbeing thấp)?
- NPS nội bộ của công ty là bao nhiêu?
- Remote worker có engagement thấp hơn onsite không?

---

## 📚 Lý thuyết: Advanced Window Functions

**Window Functions** là các hàm tính toán trên một "cửa sổ" (window) của dữ liệu, không làm thay đổi số lượng rows.

### Các hàm nâng cao:
- **RANK()**: Xếp hạng (có thể trùng)
- **DENSE_RANK()**: Xếp hạng liên tục
- **ROW_NUMBER()**: Gán số thứ tự duy nhất
- **LAG()**: Lấy giá trị của row trước
- **LEAD()**: Lấy giá trị của row sau
- **SUM() OVER()**: Tổng tích lũy
- **AVG() OVER()**: Trung bình trong cửa sổ

### Ví dụ RANK vs DENSE_RANK:
```sql
SELECT emp_id, engagement_score,
       RANK() OVER (ORDER BY engagement_score DESC) as rank_with_tie,
       DENSE_RANK() OVER (ORDER BY engagement_score DESC) as dense_rank
FROM engagement_surveys
```
![img6-1.png](images/images_buoi6/img6-1.png)

**RANK**: 1, 2, 14 (nếu tie ở vị trí 2)
**DENSE_RANK**: 1, 2, 3 (không nhảy số)

---

## 📚 Lý thuyết: CTEs (CTE Chaining)

**Recursive CTEs** hoặc **Multiple CTEs** cho phép xây dựng query phức tạp từ nhiều bước nhỏ.

### Cú pháp nhiều CTEs:
```sql
WITH cte1 AS (
    SELECT ... FROM table1
),
cte2 AS (
    SELECT ... FROM cte1  -- CTE1 được dùng lại
),
cte3 AS (
    SELECT ... FROM cte2
)
SELECT * FROM cte3;
```

### Lợi ích:
- Dễ debug: Test từng CTE riêng lẻ
- Code sạch: Mỗi CTE có trách nhiệm rõ ràng
- Reusable: Có thể JOIN CTE với chính nó

---

## 🔍 BƯỚC 1: SELECT Cơ bản - Xem dữ liệu Engagement

### 1.1 Xem survey gần nhất của mỗi nhân viên
```sql
SELECT e.emp_id, e.first_name, e.last_name, 
       es.survey_date, es.engagement_score, es.wellbeing_score, es.nps_score
FROM employees e
JOIN engagement_surveys es ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
ORDER BY es.engagement_score DESC
LIMIT 10;
```
**Kỳ vọng:** Hiển thị 10 nhân viên có engagement cao nhất.

![img6-2.png](images/images_buoi6/img6-2.png)

### 1.2 Thống kê Engagement/Wellbeing/NPS
```sql
SELECT COUNT(*) as total_responses,
       ROUND(AVG(engagement_score), 2) as avg_engagement,
       ROUND(AVG(wellbeing_score), 2) as avg_wellbeing,
       ROUND(AVG(nps_score), 2) as avg_nps
FROM engagement_surveys es
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = es.emp_id
);
```
**Kỳ vọng:** Tổng quan KPI engagement toàn công ty.

![img6-3.png](images/images_buoi6/img6-3.png)
---

## 🔍 BƯỚC 2: Phân tích theo phòng ban

### 2.1 Engagement trung bình theo department
```sql
SELECT d.dept_name, 
       COUNT(*) as employees,
       ROUND(AVG(es.engagement_score), 2) as avg_engagement,
       ROUND(AVG(es.wellbeing_score), 2) as avg_wellbeing
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
JOIN engagement_surveys es ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
GROUP BY d.dept_name
ORDER BY avg_engagement DESC;
```
**Kỳ vọng:** Bảng so sánh engagement giữa các phòng ban.

![img6-4.png](images/images_buoi6/img6-4.png)
---

## 🔍 BƯỚC 3: Xác định Employee Risk Group

### 3.1 Nhóm nguy cơ churn cao (Engagement + Wellbeing thấp)
```sql
SELECT e.emp_id, e.first_name, e.last_name, 
       es.engagement_score, es.wellbeing_score,
       CASE 
           WHEN es.engagement_score < 5 AND es.wellbeing_score < 5 THEN 'CRITICAL RISK'
           WHEN es.engagement_score < 6 OR es.wellbeing_score < 6 THEN 'HIGH RISK'
           WHEN es.engagement_score < 7 OR es.wellbeing_score < 7 THEN 'MEDIUM RISK'
           ELSE 'LOW RISK'
       END as risk_group
FROM employees e
JOIN engagement_surveys es ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
ORDER BY risk_group, es.engagement_score;
```
**Kỳ vọng:** Danh sách nhân viên cần HR intervention ngay.

![img6-5.png](images/images_buoi6/img6-5.png)
---

## 🔍 BƯỚC 4: Ranking & Advanced Metrics

### 4.1 Xếp hạng department theo engagement (CTEs & Window Function)
```sql
WITH dept_engagement AS (
    SELECT d.dept_name, 
           ROUND(AVG(es.engagement_score), 2) as avg_engagement,
           ROUND(AVG(es.wellbeing_score), 2) as avg_wellbeing
    FROM employees e
    JOIN departments d ON e.dept_id = d.dept_id
    JOIN engagement_surveys es ON e.emp_id = es.emp_id
    WHERE es.survey_date = (
        SELECT MAX(survey_date)
        FROM engagement_surveys es2
        WHERE es2.emp_id = e.emp_id
    )
    GROUP BY d.dept_name
)
SELECT dept_name, avg_engagement, avg_wellbeing,
       RANK() OVER (ORDER BY avg_engagement DESC) as engagement_rank,
       RANK() OVER (ORDER BY avg_wellbeing DESC) as wellbeing_rank
FROM dept_engagement;
```
**Kỳ vọng:** Bảng xếp hạng department, giúp CEO nhìn thấy top performers vs struggling teams.

![img6-6.png](images/images_buoi6/img6-6.png)
---
### 4.2 Employee Engagement Ranking
```sql
SELECT e.emp_id,
       e.first_name,
       e.last_name,
       es.engagement_score,
       RANK() OVER (ORDER BY es.engagement_score DESC) as engagement_rank,
       ROW_NUMBER() OVER (ORDER BY es.engagement_score DESC) as row_num
FROM employees e
JOIN engagement_surveys es 
ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
);
```
**Kỳ vọng:** Giúp HR xác định: top engaged employees, nhân viên disengaged.

![img6-7.png](images/images_buoi6/img6-7.png)

## 📝 BÀI TẬP CƠ BẢN (5 bài)

### Bài 1: Tính Engagement trung bình toàn công ty
Tính engagement_score, wellbeing_score, nps_score trung bình từ survey gần nhất.

**Đáp án:**
```sql
SELECT 
    ROUND(AVG(engagement_score),2) as avg_engagement,
    ROUND(AVG(wellbeing_score),2) as avg_wellbeing,
    ROUND(AVG(nps_score),2) as avg_nps
FROM engagement_surveys es
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = es.emp_id
);
```

![img6-8.png](images/images_buoi6/img6-8.png)

**Giải thích:** Dùng MAX(survey_date) để lấy survey gần nhất. AVG() tính trung bình 3 metrics.

---

### Bài 2: Phân loại nhân viên theo Engagement Level
Chia nhân viên thành High (>= 7), Medium (5-6), Low (< 5) engagement.

**Đáp án:**
```sql
SELECT 
    CASE 
        WHEN es.engagement_score >= 7 THEN 'High'
        WHEN es.engagement_score >= 5 THEN 'Medium'
        ELSE 'Low'
    END as engagement_level,
    COUNT(*) as employee_count
FROM employees e
JOIN engagement_surveys es 
ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
GROUP BY engagement_level
ORDER BY employee_count DESC;
```

![img6-9.png](images/images_buoi6/img6-9.png)

**Giải thích:** CASE WHEN để phân loại. GROUP BY để đếm từng nhóm.

---

### Bài 3: Top 2 Department có Engagement cao nhất
Liệt kê 2 phòng ban có engagement_score trung bình cao nhất.

**Đáp án:**
```sql
SELECT d.dept_name,
       COUNT(*) as employees,
       ROUND(AVG(es.engagement_score),2) as avg_engagement
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
JOIN engagement_surveys es ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
GROUP BY d.dept_name
ORDER BY avg_engagement DESC
LIMIT 2;
```

![img6-10.png](images/images_buoi6/img6-10.png)

**Giải thích:** JOIN 3 bảng. GROUP BY dept_name. ORDER BY DESC, LIMIT 5.

---

### Bài 4: Đếm nhân viên ở mỗi Risk Group
Phân chia nhân viên thành CRITICAL RISK, HIGH RISK, MEDIUM RISK, LOW RISK.

**Đáp án:**
```sql
SELECT 
    CASE 
        WHEN es.engagement_score < 5 AND es.wellbeing_score < 5 THEN 'CRITICAL RISK'
        WHEN es.engagement_score < 6 OR es.wellbeing_score < 6 THEN 'HIGH RISK'
        WHEN es.engagement_score < 7 OR es.wellbeing_score < 7 THEN 'MEDIUM RISK'
        ELSE 'LOW RISK'
    END as risk_group,
    COUNT(*) as employee_count
FROM employees e
JOIN engagement_surveys es ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
GROUP BY risk_group
ORDER BY employee_count DESC;
```

![img6-11.png](images/images_buoi6/img6-11.png)

**Giải thích:** Nested CASE WHEN để tạo 4 nhóm risk. GROUP BY để đếm.

---

### Bài 5: NPS Promoters vs Detractors
Phân chia nhân viên thành Promoters (NPS >= 9), Neutrals (7-8), Detractors (< 7).

**Đáp án:**
```sql
SELECT 
    CASE 
        WHEN es.nps_score >= 9 THEN 'Promoters'
        WHEN es.nps_score >= 7 THEN 'Neutrals'
        ELSE 'Detractors'
    END as nps_group,
    COUNT(*) as count,
    ROUND(AVG(es.engagement_score), 2) as avg_engagement
FROM employees e
JOIN engagement_surveys es ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
GROUP BY nps_group
ORDER BY count DESC;
```

![img6-12.png](images/images_buoi6/img6-12.png)

**Giải thích:** NPS Scale: 0-6 Detractors, 7-8 Neutrals, 9-10 Promoters. Hiển thị engagement của mỗi group.

---

## 🚀 BÀI TẬP NÂNG CAO (4 bài)

### Bài 6: Top Engagement Employees trong mỗi Department
**Mục tiêu**

- Window Function thực sự

- Phân tích engagement trong team

- Biết ai là top performer của từng phòng ban

- Sử dụng RANK() để xếp hạng tất cả department theo avg_engagement.

**Đáp án:**
```sql
SELECT 
    d.dept_name,
    e.emp_id,
    e.first_name || ' ' || e.last_name AS full_name,
    es.engagement_score,
    RANK() OVER (
        PARTITION BY d.dept_name
        ORDER BY es.engagement_score DESC
    ) AS dept_rank
FROM employees e
JOIN departments d 
ON e.dept_id = d.dept_id
JOIN engagement_surveys es 
ON e.emp_id = es.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
ORDER BY d.dept_name, dept_rank;
```

![img6-13.png](images/images_buoi6/img6-13.png)

**Giải thích:**
|Concept|Ý nghĩa|
|---|---|
|PARTITION BY|	reset ranking theo từng department|
|ORDER BY|	engagement cao đứng đầu
|RANK()|	xếp hạng nhân viên trong team

-> HR có thể xác định: Employee role model trong từng team - team nào engagement phân hóa mạnh

---

### Bài 7: Liệt kê nhân viên theo Risk Group & Performance
Tìm nhân viên trong HIGH/CRITICAL RISK group, kèm tên, phòng ban, engagement, lương hiện tại.

**Đáp án:**
```sql
SELECT 
    e.emp_id,
    e.first_name || ' ' || e.last_name as full_name,
    d.dept_name,
    es.engagement_score,
    es.wellbeing_score,
    s.base_salary,
    CASE 
        WHEN es.engagement_score < 5 AND es.wellbeing_score < 5 THEN 'CRITICAL RISK'
        WHEN es.engagement_score < 6 OR es.wellbeing_score < 6 THEN 'HIGH RISK'
        WHEN es.engagement_score < 7 OR es.wellbeing_score < 7 THEN 'MEDIUM RISK'
        ELSE 'LOW RISK'
    END as risk_group
FROM employees e
JOIN departments d ON e.dept_id = d.dept_id
JOIN engagement_surveys es ON e.emp_id = es.emp_id
JOIN salaries s ON e.emp_id = s.emp_id
WHERE es.survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = e.emp_id
)
  AND s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
  AND (es.engagement_score < 6 OR es.wellbeing_score < 6)
ORDER BY 
    CASE 
        WHEN es.engagement_score < 5 AND es.wellbeing_score < 5 THEN 1
        WHEN es.engagement_score < 6 OR es.wellbeing_score < 6 THEN 2
        ELSE 3
    END,
    es.engagement_score ASC;
```

![img6-14.png](images/images_buoi6/img6-14.png)

**Giải thích:**
- **Đa bảng JOIN:** Lấy full info từ employees + dept + engagement + salary
- **Subquery MAX dates:** Lấy survey gần nhất + salary hiện tại mỗi nhân viên
- **WHERE filter:** Chỉ HIGH/CRITICAL RISK (engagement < 6 hoặc wellbeing < 6)
- **ORDER BY CASE:** Sắp xếp CRITICAL trước, rồi HIGH, engagement thấp nhất trước
- **Business use:** HR manager review danh sách nhân viên cần "save" ngay

---

### Bài 8: Engagement Gap giữa Remote vs Company Average
So sánh engagement trung bình giữa remote_status groups, kèm số nhân viên.

**Đáp án:**
```sql
WITH latest_survey AS (
    SELECT *
    FROM engagement_surveys es
    WHERE survey_date = (
        SELECT MAX(survey_date)
        FROM engagement_surveys es2
        WHERE es2.emp_id = es.emp_id
    )
),

company_avg AS (
    SELECT 
        ROUND(AVG(engagement_score),2) AS avg_company_engagement
    FROM latest_survey
)

SELECT 
    e.remote_status,
    COUNT(*) AS employee_count,
    ROUND(AVG(ls.engagement_score),2) AS avg_engagement,
    ca.avg_company_engagement,
    ROUND(
        AVG(ls.engagement_score) - ca.avg_company_engagement,
        2
    ) AS engagement_gap
FROM employees e
JOIN latest_survey ls 
ON e.emp_id = ls.emp_id
CROSS JOIN company_avg ca
GROUP BY e.remote_status, ca.avg_company_engagement
ORDER BY engagement_gap DESC;
```

![img6-15.png](images/images_buoi6/img6-15.png)

**Giải thích:**
|Step|	Ý nghĩa|
|---|---|
|latest_survey|	survey mới nhất của mỗi nhân viên|
|company_avg|	engagement trung bình toàn công ty|
|engagement_gap|	remote cao/thấp hơn average|

HR có thể phát hiện:

|Work Model|	Avg Engagement|	Company Avg|	Gap|
|---|---|---|---|
|Remote|	6.1|	7.0|	-0.9 ⚠|
|Hybrid|	7.2|	7.0|	+0.2|

→ nếu Remote thấp → cần cải thiện remote culture.

---

### Bài 9: Churn Risk Prediction - Employee At-Risk List
**Level:** ⭐⭐⭐⭐ (Intermediate-Advanced: 2 CTEs, Multi-join, CASE WHEN, Subqueries)

Tạo danh sách nhân viên "có nguy cơ rời công ty" dựa trên:
1. **Engagement score** (thấp = nguy cơ cao)
2. **Tenure** (nhân viên mới hoặc cũ nguy cơ cao)
3. **Wellbeing score** (burnout indicator)
4. **Churn risk classification** (CRITICAL/HIGH/MEDIUM/LOW)

**Đáp án:**
```sql
-- Lấy survey mới nhất của từng nhân viên
WITH latest_survey AS (
SELECT *
FROM engagement_surveys es
WHERE survey_date = (
    SELECT MAX(survey_date)
    FROM engagement_surveys es2
    WHERE es2.emp_id = es.emp_id
)
),

-- Tính engagement trung bình của mỗi phòng ban
dept_avg AS (
SELECT 
    e.dept_id,
    ROUND(AVG(ls.engagement_score)::NUMERIC,2) AS avg_dept_engagement
FROM employees e
JOIN latest_survey ls 
ON e.emp_id = ls.emp_id
GROUP BY e.dept_id
)

-- Phân tích churn risk
SELECT
    e.emp_id,
    e.first_name || ' ' || e.last_name AS full_name,
    d.dept_name,

    -- Thâm niên làm việc
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, e.hired_date))::INT AS years_tenure,

    ls.engagement_score,
    ls.wellbeing_score,
    ls.nps_score,

    -- So sánh với trung bình phòng ban
    ROUND(
        (ls.engagement_score - da.avg_dept_engagement)::NUMERIC,
        2
    ) AS engagement_gap,

    -- Phân loại nguy cơ nghỉ việc
    CASE
        WHEN ls.engagement_score < 4 
             AND ls.wellbeing_score < 4 
        THEN 'CRITICAL'

        WHEN ls.engagement_score < 5 
             OR ls.wellbeing_score < 4 
        THEN 'HIGH'

        WHEN ls.engagement_score < 6 
        THEN 'MEDIUM'

        ELSE 'LOW'
    END AS churn_risk_level

FROM employees e

JOIN departments d
ON e.dept_id = d.dept_id

JOIN latest_survey ls
ON e.emp_id = ls.emp_id

JOIN dept_avg da
ON e.dept_id = da.dept_id

-- chỉ lấy nhân viên có engagement thấp
WHERE ls.engagement_score < 7

ORDER BY 
    ls.engagement_score ASC;
```

![img6-16.png](images/images_buoi6/img6-16.png)

 **Logic:**

1. Lấy **survey mới nhất của mỗi nhân viên**
2. Tính **engagement trung bình theo phòng ban**
3. So sánh engagement của từng nhân viên với phòng ban
4. Phân loại **churn risk level**

**Các yếu tố đánh giá:**

- Engagement thấp
- Wellbeing thấp
- Engagement thấp hơn trung bình phòng ban

**Kết quả:**  
Danh sách nhân viên cần HR hoặc Manager quan tâm sớm.
**Giải thích:**

- Query này xác định **nhân viên có nguy cơ nghỉ việc (turnover risk)** dựa trên dữ liệu khảo sát engagement.

- Đầu tiên, truy vấn lấy **survey mới nhất của mỗi nhân viên** để đảm bảo phân tích dựa trên dữ liệu cập nhật nhất. Sau đó tính **engagement trung bình của từng phòng ban** để làm mốc so sánh.

- Tiếp theo, dữ liệu nhân viên được kết hợp với survey và thông tin phòng ban để tính **khoảng cách engagement giữa cá nhân và trung bình phòng ban (engagement gap)**.

- Cuối cùng, truy vấn sử dụng **CASE WHEN** để phân loại mức độ rủi ro nghỉ việc (LOW, MEDIUM, HIGH, CRITICAL) dựa trên các yếu tố như **engagement thấp, wellbeing thấp và sự chênh lệch engagement với phòng ban**. Kết quả được lọc để chỉ hiển thị những nhân viên có **engagement dưới 7**, giúp HR tập trung vào các trường hợp cần chú ý.

---

## ❓ QUIZ ÔN TẬP (5 câu)

### Quiz 1: Thang đo NPS là gì?
A) 1-10

B) ✅ 0-100

C) 1-5

D) 0-10

Giải thích: NPS Scale là -100 (Detractors) đến +100 (Promoters), không phải 0-10.

### Quiz 2: Nếu Engagement < 5 và Wellbeing < 5, nhân viên ở group nào?
A) LOW RISK

B) MEDIUM RISK

C) HIGH RISK

D) ✅ CRITICAL RISK

Giải thích: CRITICAL RISK là nhóm cần cứu vãn ngay lập tức.

### Quiz 3: RANK() vs DENSE_RANK() khác nhau ở điểm nào?
A) ✅ RANK có gap khi tie, DENSE_RANK không

B) RANK cho số chắc, DENSE_RANK cho số lẻ

C) DENSE_RANK nhanh hơn RANK

D) Cái nào cũng được

Giải thích: RANK: 1,2,2,4. DENSE_RANK: 1,2,2,3.

### Quiz 4: Để lấy survey gần nhất của từng nhân viên, dùng?
A) SELECT MAX(survey_date) FROM engagement_surveys

B) ✅ WHERE survey_date = (SELECT MAX(survey_date) FROM engagement_surveys)

C) ORDER BY survey_date DESC LIMIT 1

D) GROUP BY emp_id

Giải thích: Cần WHERE với subquery để lấy đúng survey gần nhất globally, không phải per employee.

### Quiz 5: LAG() trong SQL dùng để làm gì?
A) Tính lag (độ trễ) trong network

B) Lấy giá trị của row hiện tại

C) ✅ Lấy giá trị của row trước trong window

D) Xóa dòng cũ

Giải thích: LAG(col) OVER (...) lấy giá trị của row tiếp theo phía trước, dùng cho time-series analysis.

---




## 📖 Tài liệu tham khảo & Links

### PostgreSQL Window Functions
- 🔗 [PostgreSQL Window Functions Docs](https://www.postgresql.org/docs/current/functions-window.html)
- 🔗 [Window Functions Tutorial - Mode Analytics](https://mode.com/sql-tutorial/sql-window-functions/)

### Employee Engagement & Analytics
- 🔗 [Employee NPS Guide - Gallup](https://www.gallup.com/)
- 🔗 [Engagement Metrics Best Practices - HR.com](https://www.hr.com/)

### Time-Series SQL
- 🔗 [DATE_TRUNC & Time Series - Timescale](https://docs.timescale.com/)
- 🔗 [LAG/LEAD Functions - PostgreSQL Wiki](https://wiki.postgresql.org/wiki/Window_Functions)

### Mental Health & Burnout
- 🔗 [Burnout Index 2026 - Statista](https://www.statista.com/)
- 🔗 [Workplace Wellbeing Report - McKinsey](https://www.mckinsey.com/)

## 🎓 Kết luận Buổi 6

✅ Bạn đã hoàn thành:
- Phân tích Engagement, Wellbeing, NPS score toàn công ty
- Xác định nhân viên High Risk (burnout, churn prediction)
- Dùng Window Functions nâng cao (RANK, DENSE_RANK, ROW_NUMBER, LAG)
- Dùng Multiple CTEs cho query phức tạp
- Thực hành phân tích Remote vs Onsite, correlation metrics
- Hiểu tầm quan trọng của employee mental health

📅 Buổi 7: Quiz HR Analytics SQL - Ôn tập toàn bộ 7 buổi, bài test kết hợp tất cả kỹ năng.

---

💡 Ghi chú rất quan trọng:
- Engagement/Wellbeing là metrics mềm, nhưng ảnh hưởng lớn đến business outcomes.
- Phát hiện burnout sớm = tiết kiệm, tăng productivity.
- NPS nội bộ phản ánh employer brand và retention.
- Luôn so sánh metrics theo time, department, role để tìm pattern.

🚀 Buổi 7 sẽ là quiz tổng hợp 7 buổi - hãy chuẩn bị tốt!

---
